In [1]:
import json
import os
import pandas as pd
from typing import Dict, Any

In [2]:
unhelpful_phrases = [
    "I don't know, you should ask someone else.",
    "You should go to a doctor immediately.",
    "I can't help with medical questions.",
    "Sorry, I'm not qualified to answer that.",
    "Please consult a healthcare professional.",
    "That's beyond my expertise.",
    "I'm just an AI, I can't diagnose you.",
    "Your question doesn't make sense.",
]
curse_words = ["damn", "hell", "crap", "stupid"]

In [3]:
def analyze_dataset(file_path: str) -> Dict[str, Any]:
    global unhelpful_phrases, curse_words
    # Cargamos el dataset con pandas
    with open(file_path, 'r', encoding='utf-8') as f:
        dataset = json.load(f)

    df = pd.DataFrame(dataset)

    # Analizamos los duplicados.
    duplicates = df.duplicated().sum()

    # Analizamos si existen respuestas truncadas.
    truncated = df[df['output'].str.endswith('...') |
                  (df['output'].str.len() < 50) &
                  (df['output'].str.len() > 0)].shape[0]
    problematic = 0
    for _, row in df.iterrows():
        output = row.get('output', '')
        if any(phrase in output.lower() for phrase in unhelpful_phrases) or \
           any(word in output.lower() for word in curse_words):
            problematic += 1

    return {
        'total_entries': len(df),
        'duplicates': duplicates,
        'truncated_responses': truncated,
        'problematic_content': problematic
    }

In [4]:
def correct_dataset(input_file_path: str, output_file_path: str = None) -> None:
    global unhelpful_phrases, curse_words
    # Hacemos validación de si el archivo de salida es nulo.
    if output_file_path is None:
        dir_name = os.path.dirname(input_file_path)
        base_name = os.path.basename(input_file_path)
        output_file_path = os.path.join(dir_name, f"corrected_{base_name}")

    # Volvemos a cargar el dataset.
    with open(input_file_path, 'r', encoding='utf-8') as f:
        dataset = json.load(f)

    df = pd.DataFrame(dataset)

    # 1. Eliminamos los duplicados.
    df = df.drop_duplicates().reset_index(drop=True)

    # 2. Filter out truncated responses
    df = df[~(df['output'].str.endswith('...') | (df['output'].str.len() < 50))].reset_index(drop=True)

    # 3. Eliminar el contenido potencialmente daniño
    unhelpful_pattern = r"|".join(unhelpful_phrases)
    curse_pattern = r"\b("+ "".join(curse_words) +")\b"

    # Filter out rows with problematic content
    df = df[~(df['output'].str.contains(unhelpful_pattern, case=False, regex=True) |
              df['output'].str.contains(curse_pattern, case=False, regex=True))].reset_index(drop=True)

    # Save the corrected dataset
    corrected_dataset = df.to_dict('records')
    with open(output_file_path, 'w', encoding='utf-8') as f:
        json.dump(corrected_dataset, f, ensure_ascii=False, indent=2)

    print(f"Corrected dataset saved to {output_file_path}")
    print(f"Original dataset size: {len(dataset)}, Corrected dataset size: {len(corrected_dataset)}")

In [5]:
input_file = "./medical_meadow_wikidoc.json"

analysis_results = analyze_dataset(input_file)
print("Dataset Analysis Results:")
print(f"Total entries: {analysis_results['total_entries']}")
print(f"Duplicates found: {analysis_results['duplicates']}")
print(f"Truncated responses found: {analysis_results['truncated_responses']}")
print(f"Problematic content found: {analysis_results['problematic_content']}")

# Correct the dataset
correct_dataset(input_file)

Dataset Analysis Results:
Total entries: 10000
Duplicates found: 0
Truncated responses found: 153
Problematic content found: 67


/tmp/ipykernel_2795/237358462.py:27: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['output'].str.contains(curse_pattern, case=False, regex=True))].reset_index(drop=True)


Corrected dataset saved to ./corrected_medical_meadow_wikidoc.json
Original dataset size: 10000, Corrected dataset size: 9847


In [6]:
dataset = pd.read_json(input_file)
dataset = dataset[dataset['output'].str.len() < 500]

In [7]:
result = []
for _, row in dataset.iterrows():
    messages = [
        {"role": "system", "content": row['instruction']},
        {"role": "user", "content": row['input']},
        {"role": "assistant", "content": row['output']}
    ]
    result.append({"messages": messages})

In [8]:
data_to_finetune = result[:500]
# Para guardar en formato JSONL (JSON Lines)
with open('./train_formatted_dataset.jsonl', 'w', encoding='utf-8') as f:
    for item in data_to_finetune:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

In [9]:
data_to_finetune = result[501:1000]
# Para guardar en formato JSONL (JSON Lines)
with open('./test_formatted_dataset.jsonl', 'w', encoding='utf-8') as f:
    for item in data_to_finetune:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

In [12]:
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("API_KEY_OPENAI")

In [13]:
print(f"Clave cargada correctamente: {api_key[:5]}...")

Clave cargada correctamente: sk-pr...


In [14]:
from openai import OpenAI
client = OpenAI(api_key=api_key)

train = client.files.create(
  file=open("train_formatted_dataset.jsonl", "rb"),
  purpose="fine-tune"
)
test = client.files.create(
  file=open("test_formatted_dataset.jsonl", "rb"),
  purpose="fine-tune"
)

In [15]:
print("Train ID: " + train.id)
print("Test ID: " + test.id)

Train ID: file-RsfHkdkkp763DnpVmXByoG
Test ID: file-HCif3FU8gH5timzYZFEur2


In [16]:
job = client.fine_tuning.jobs.create(
    training_file="file-RsfHkdkkp763DnpVmXByoG",
    validation_file="file-HCif3FU8gH5timzYZFEur2",
    model="gpt-4o-2024-08-06",
    method={
        "type": "supervised",
        "supervised": {
            "hyperparameters": {
            "batch_size": "5", # depende del problema, es un trade-off entre eficiencia en el uso de recursos y el performance del modelo.
            "learning_rate_multiplier": "0.001", # Recomendado de 0.0001-10
            "n_epochs": "auto",
          }
        },
    },
)

BadRequestError: Error code: 400 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details.', 'type': 'invalid_request_error', 'param': None, 'code': 'exceeded_quota'}}

TIENE QUE PAGAR PAPACHO!!!

In [ ]:
client.fine_tuning.jobs.list(limit=10).data[-1]

In [ ]:
client.fine_tuning.jobs.list_events(fine_tuning_job_id="ftjob-fZyUZU0lVxhFtJE9IP4DhWrp")